# w22_cpu1_apa_leadtime_and_delayed_onset_4dataset
Inputs: `shortpose-wave1-private-inputs` | Accelerator: `None` (Kaggle 4-vCPU Cloud)


In [ ]:
import hashlib, os, subprocess, sys, tempfile, zipfile
from pathlib import Path

WORK, MOUNT = Path('/kaggle/working'), Path('/kaggle/input')
assert WORK.is_dir(), 'Kaggle Cloud environment required'
REV = '26aef1d2665cea0491c0f7498c40884aca1421d3'
with tempfile.TemporaryDirectory(prefix='w22_c1_') as ts:
    tmp = Path(ts)
    repo, meta_dir, sc0, sc1 = tmp / 'repo', tmp / 'pinns_meta', tmp / 'sc0', tmp / 'sc1'
    subprocess.run(['git', 'init', '-q', str(repo)], check=True)
    subprocess.run(['git', '-C', str(repo), 'fetch', '-q', '--depth=1', 'https://github.com/pjyyyds/Unstructured-ShortPose-Pred.git', REV], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', '-q', '--detach', 'FETCH_HEAD'], check=True)
    env2 = {**os.environ, 'PYTHONPATH': str(repo), 'PYTHONDONTWRITEBYTECODE': '1', 'OMP_NUM_THREADS': '2', 'MKL_NUM_THREADS': '2'}
    joint_bin = next(MOUNT.rglob('joint_actions_all.bin'))
    heldout_bin = next(MOUNT.rglob('shortpose_heldout.bin'))
    map_bin = next(MOUNT.rglob('pinns_source_label_causal_maps.bin'))
    pose_npz = tmp / 'pose.npz'
    with zipfile.ZipFile(heldout_bin) as zf:
        pose_npz.write_bytes(zf.read('pose.npz'))
    subprocess.run([sys.executable, '-m', 'scripts.stage_pinns_metadata', '--output-dir', str(meta_dir)], cwd=repo, env=env2, check=True)
    d0 = WORK / 'wave22_apa_leadtime_jaad_jta_and_pinns_delayed.zip'
    d1 = WORK / 'wave22_jrdb_delayed_apa_upgrade.zip'
    p0 = subprocess.Popen([
        sys.executable, '-u', '-m', 'scripts.run_wave22_apa_leadtime_and_delayed_onset_4dataset',
        '--part', 'gpu0_jaad_jta_pinns', '--scratch-dir', str(sc0),
        '--meta-dir', str(meta_dir), '--pose-npz', str(pose_npz),
        '--joint-zip', str(joint_bin), '--causal-map-bin', str(map_bin), '--zip-path', str(d0)
    ], cwd=repo, env=env2)
    p1 = subprocess.Popen([
        sys.executable, '-u', '-m', 'scripts.run_wave22_apa_leadtime_and_delayed_onset_4dataset',
        '--part', 'gpu1_jrdb', '--scratch-dir', str(sc1), '--zip-path', str(d1)
    ], cwd=repo, env=env2)
    rc0, rc1 = p0.wait(), p1.wait()
    assert rc0 == 0 and rc1 == 0, f'Subprocess failed: rc0={rc0}, rc1={rc1}'
    for d in (d0, d1):
        print('DONE:', d, d.stat().st_size, hashlib.sha256(d.read_bytes()).hexdigest(), flush=True)
